# Notebook 16: blind-set catalogue neighbours

Rank 19 on the live board is running our architecture (chemprop D-MPNN, ChEMBL plus
AID 1851 as auxiliary heads, affine calibration at k = rho) and scores 0.4378 against
our 0.5064. Rank 28 does one thing neither of us does: it pulls tens of thousands of
near-neighbours of the 750 blind structures out of a public catalogue and puts them in
the training pool carrying calculated physical properties and no assay labels.

That attacks a gap we measured ourselves in September. Blind median Tanimoto to training
is 0.578; our holdout sits at 0.435. We validate on chemistry harder than the blind set,
and no model of ours has ever seen the neighbourhood it gets scored in.

This notebook retrieves those neighbours from PubChem, screens them, and adds them as
extra rows with physicochemical heads only.

**Runtime** roughly 2.5 to 4 hours on a T4. Retrieval is about 20 to 40 minutes of it and
checkpoints every 25 compounds, so a disconnect costs almost nothing. Every stage degrades
to a no-op rather than an exception: if PubChem is unreachable the notebook still runs and
reproduces the notebook 5 baseline, which is the control we want anyway.


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable:', type(e).__name__)
os.makedirs(OUTDIR, exist_ok=True)
print('writing to', OUTDIR)

In [ ]:
!pip -q install chemprop==2.3.1

In [ ]:
import gc, json, time, urllib.request, numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy'); G=np.load(CACHE/'scaffold_groups.npy',allow_pickle=True)
AUX=np.load(CACHE/'aux_train.npy')
SM_TR=meta['SMILES'].to_numpy(); SM_TE=mete['SMILES'].to_numpy()

CK='/content/chemprop_medium.pt'
if not Path(CK).exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/jeremycheminf/'
        'openadmet_scripts/main/CYP_Challenge/checkpoints/chemprop_medium.pt', CK)
print('checkpoint', round(Path(CK).stat().st_size/1e6,1), 'MB')

URL=('https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download'
     '&record_type=datatable&actvty=all&response_type=save&aid=1851')
AID = OUT / 'aid1851.csv'
if not AID.exists():
    print('downloading AID 1851 (first run only; cached on Drive after this)')
    urllib.request.urlretrieve(URL, str(AID))
raw=pd.read_csv(AID, low_memory=False, skiprows=[1])
raw['pac50']=-pd.to_numeric(raw['Fit_LogAC50'], errors='coerce')
PANEL={'p450-cyp1a2':'CYP1A2','p450-cyp2c9':'CYP2C9','p450-cyp2c19':'CYP2C19',
       'p450-cyp2d6':'CYP2D6','p450-cyp3a4':'CYP3A4'}
raw['panel']=raw['Panel Name'].map(PANEL)
raw=raw.dropna(subset=['panel','pac50','PUBCHEM_EXT_DATASOURCE_SMILES'])
print('qHTS measurements', len(raw))

## Stage 1: retrieve catalogue neighbours of the blind set

One PubChem 2D fast-similarity query per blind compound, at a Tanimoto threshold of 80%,
up to 200 records each. Checkpointed to Drive every 25 compounds; re-running the cell
resumes. PubChem renamed this property field between API versions, so the SMILES column
is located by name rather than assumed.


In [ ]:
import urllib.request, urllib.parse, urllib.error

NEI_RAW = OUT / 'nb16_neighbours_raw.json'
THRESH, MAXREC = 80, 200
PUG = ('https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/fastsimilarity_2d/'
       'smiles/property/CanonicalSMILES/JSON?Threshold=%d&MaxRecords=%d')

def _smiles_of(rec):
    # PubChem has renamed this field across API versions (CanonicalSMILES,
    # ConnectivitySMILES, SMILES), so find it by name instead of assuming one.
    for k, v in rec.items():
        if 'SMILES' in k.upper() and isinstance(v, str) and v:
            return v
    return None

def query(smi, tries=3):
    """Returns a list of neighbour SMILES, or None if the query never succeeded."""
    body = urllib.parse.urlencode({'smiles': smi}).encode()
    url = PUG % (THRESH, MAXREC)
    for a in range(tries):
        try:
            req = urllib.request.Request(url, data=body)
            with urllib.request.urlopen(req, timeout=180) as r:
                js = json.load(r)
            props = js.get('PropertyTable', {}).get('Properties', [])
            return [s for s in (_smiles_of(p) for p in props) if s]
        except urllib.error.HTTPError as e:
            if e.code in (400, 404):      # no hits, or PubChem cannot parse the SMILES
                return []
            time.sleep(3 + 4 * a)
        except Exception:
            time.sleep(3 + 4 * a)
    return None

store = {}
if NEI_RAW.exists():
    try:
        store = json.load(open(NEI_RAW))
    except Exception as e:
        print('checkpoint unreadable, starting over:', type(e).__name__)
print(f'resuming with {len(store)} of {len(SM_TE)} blind compounds already queried')

t0, new, failed = time.time(), 0, 0
for smi in SM_TE:
    if smi in store:
        continue
    got = query(smi)
    if got is None:
        failed += 1
        got = []
    store[smi] = got
    new += 1
    if new % 25 == 0:
        json.dump(store, open(NEI_RAW, 'w'))
        print(f'  {len(store)}/{len(SM_TE)} queried, '
              f'{sum(len(v) for v in store.values())} hits, {failed} failed '
              f'({time.time() - t0:.0f}s)', flush=True)
    time.sleep(0.25)
json.dump(store, open(NEI_RAW, 'w'))

CAND = sorted({s for v in store.values() for s in v})
print(f'\nqueried {len(store)} blind compounds, {failed} failed after retries')
print('unique candidate neighbours:', len(CAND))
if len(CAND) < 500:
    print('\n*** PubChem returned almost nothing. The notebook will carry on with no '
          'neighbours, which reproduces the notebook 5 baseline. That is still a useful '
          'control, but the experiment itself has not run. ***')


## Stage 2: screen them

Three filters, in increasing order of cost so the expensive one sees the fewest molecules.

1. **Physicochemical envelope.** Molecular weight, logP, TPSA and rotatable bonds must all
   fall inside the blind set's own observed range.
2. **Similarity floor.** A candidate's best Tanimoto to any blind compound must exceed the
   training set's own average nearest-neighbour similarity, so every admitted molecule is
   closer to the scored compounds than our training data is to itself.
3. **Structural alerts, vetoed by the blind set.** PAINS, BRENK and NIH, applied
   subtractively: any alert that fires on even one blind compound is disqualified as a
   reason to reject a candidate, because the scored set evidently contains that motif.

Anything already in our training or blind sets is dropped on canonical SMILES.


In [ ]:
from rdkit import DataStructs
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.Chem.MolStandardize import rdMolStandardize

N_KEEP = 5000
LFC = rdMolStandardize.LargestFragmentChooser()
MFG = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def mol(s):
    try:
        return Chem.MolFromSmiles(s) if isinstance(s, str) else None
    except Exception:
        return None

def canon(s):
    m = mol(s)
    return Chem.MolToSmiles(m) if m is not None else None

def envelope(m):
    return (Descriptors.MolWt(m), Crippen.MolLogP(m),
            rdMolDescriptors.CalcTPSA(m), rdMolDescriptors.CalcNumRotatableBonds(m))

te_mols = [m for m in map(mol, SM_TE) if m is not None]
te_fps = [MFG.GetFingerprint(m) for m in te_mols]
env = np.array([envelope(m) for m in te_mols], float)
LO, HI = env.min(0), env.max(0)
print('blind envelope  MW %.0f-%.0f  logP %.2f-%.2f  TPSA %.0f-%.0f  rotB %.0f-%.0f'
      % (LO[0], HI[0], LO[1], HI[1], LO[2], HI[2], LO[3], HI[3]))

params = FilterCatalogParams()
for c in (FilterCatalogParams.FilterCatalogs.PAINS,
          FilterCatalogParams.FilterCatalogs.BRENK,
          FilterCatalogParams.FilterCatalogs.NIH):
    params.AddCatalog(c)
CAT = FilterCatalog(params)
BLIND_ALERTS = set()
for m in te_mols:
    for e in CAT.GetMatches(m):
        BLIND_ALERTS.add(e.GetDescription())
print('alerts disqualified by firing on the blind set:', len(BLIND_ALERTS))

tr_mols = [m for m in map(mol, SM_TR) if m is not None]
tr_fps = [MFG.GetFingerprint(m) for m in tr_mols]
_nn = []
for i, fp in enumerate(tr_fps):
    sims = DataStructs.BulkTanimotoSimilarity(fp, tr_fps)
    sims[i] = -1.0
    _nn.append(max(sims))
FLOOR = float(np.mean(_nn))
print('training set average nearest-neighbour Tanimoto: %.4f' % FLOOR)

NEI_CACHE = OUT / 'nb16_neighbours.json'
if NEI_CACHE.exists():
    NEI = np.array(json.load(open(NEI_CACHE)), dtype=object)
    print(f'loaded {len(NEI)} screened neighbours from the checkpoint; '
          'skipping the screen')
else:
    OURS = {c for c in map(canon, np.concatenate([SM_TR, SM_TE])) if c}

    # PubChem returns salt and mixture forms; our own SMILES are single components, and a
    # dot-disconnected string featurises as a different molecule. Keep the largest fragment,
    # then require the elements to be ones our own data actually contains.
    ELEMS = {a.GetSymbol() for m in te_mols for a in m.GetAtoms()}
    ELEMS |= {a.GetSymbol() for m in tr_mols for a in m.GetAtoms()}
    print('element alphabet from our own compounds:', ' '.join(sorted(ELEMS)))

    kept, n_env, n_sim, n_alert, n_dup, n_elem = [], 0, 0, 0, 0, 0
    for n, s in enumerate(CAND):
        if n and n % 10000 == 0:
            print(f'  screened {n}/{len(CAND)}, {len(kept)} kept so far', flush=True)
        m = mol(s)
        if m is None or m.GetNumAtoms() == 0:
            continue
        try:
            m = LFC.choose(m)
        except Exception:
            continue
        if m is None or m.GetNumAtoms() == 0:
            continue
        if any(a.GetSymbol() not in ELEMS for a in m.GetAtoms()):
            n_elem += 1
            continue
        p = envelope(m)
        if any(p[k] < LO[k] or p[k] > HI[k] for k in range(4)):
            n_env += 1
            continue
        smx = max(DataStructs.BulkTanimotoSimilarity(MFG.GetFingerprint(m), te_fps))
        if smx < FLOOR:
            n_sim += 1
            continue
        if any(e.GetDescription() not in BLIND_ALERTS for e in CAT.GetMatches(m)):
            n_alert += 1
            continue
        c = Chem.MolToSmiles(m)          # the stripped molecule, not the catalogue string
        if not c or c in OURS:
            n_dup += 1
            continue
        kept.append((c, smx))

    kept.sort(key=lambda r: -r[1])
    seen, NEI, NEI_SIM = set(), [], []
    for c, smx in kept:
        if c in seen:
            continue
        seen.add(c)
        NEI.append(c)
        NEI_SIM.append(smx)
        if len(NEI) >= N_KEEP:
            break
    NEI = np.array(NEI, dtype=object)
    print(f'\nrejected: element {n_elem}, envelope {n_env}, similarity {n_sim}, '
          f'alerts {n_alert}, duplicate {n_dup}')
    print(f'admitted {len(NEI)} neighbours')
    if len(NEI):
        print('similarity to the blind set: max %.3f, median %.3f, min %.3f'
              % (NEI_SIM[0], float(np.median(NEI_SIM[:len(NEI)])), NEI_SIM[len(NEI) - 1]))
    json.dump([str(x) for x in NEI], open(NEI_CACHE, 'w'))


## Stage 3: the target table

The thirteen heads from notebook 5 (four challenge isoforms, four single-concentration
screen arms, five qHTS panels) plus ten calculated physicochemical heads that every row
carries, neighbours included.

Those calculated heads are **down-weighted**, following rank 28's note that a property
computable from structure alone carries a label on every row and at full weight supplies
most of the pretraining gradient, so the trunk spends its capacity reproducing a function
it already has. A constant per-row weight would cancel out of a weighted mean, so this is
a per-task weight.


In [ ]:
ours_set = {c for c in map(canon, np.concatenate([SM_TR, SM_TE])) if c}
pub = raw.groupby(['PUBCHEM_EXT_DATASOURCE_SMILES', 'panel'], as_index=False).pac50.median()
pub['canon'] = [canon(s) for s in pub.PUBCHEM_EXT_DATASOURCE_SMILES]
pub = pub.dropna(subset=['canon'])
pub = pub[~pub.canon.isin(ours_set)]
wide = pub.pivot_table(index='PUBCHEM_EXT_DATASOURCE_SMILES', columns='panel', values='pac50')
PAN = ['CYP1A2', 'CYP2C9', 'CYP2C19', 'CYP2D6', 'CYP3A4']
wide = wide.reindex(columns=PAN)
print('public qHTS compounds after dropping overlaps', len(wide))

DESC = {
    'MolWt': Descriptors.MolWt,
    'MolLogP': Crippen.MolLogP,
    'TPSA': rdMolDescriptors.CalcTPSA,
    'NumRotatableBonds': rdMolDescriptors.CalcNumRotatableBonds,
    'NumHAcceptors': rdMolDescriptors.CalcNumHBA,
    'NumHDonors': rdMolDescriptors.CalcNumHBD,
    'RingCount': rdMolDescriptors.CalcNumRings,
    'FractionCSP3': rdMolDescriptors.CalcFractionCSP3,
    'NumAromaticRings': rdMolDescriptors.CalcNumAromaticRings,
    'HeavyAtomCount': lambda m: float(m.GetNumHeavyAtoms()),
}
DNAMES = list(DESC)
ASSAY = [f'chal_{i}' for i in ISO] + [f'scr_{i}' for i in ISO] + [f'qh_{p}' for p in PAN]
HEADS = ASSAY + [f'desc_{d}' for d in DNAMES]
DESC_W = 0.1
HEAD_W = np.array([1.0] * len(ASSAY) + [DESC_W] * len(DNAMES), np.float32)

n_ch, n_pub, n_nei = len(SM_TR), len(wide), len(NEI)
SM_ALL = np.concatenate([SM_TR, wide.index.to_numpy(), NEI]) if n_nei else \
         np.concatenate([SM_TR, wide.index.to_numpy()])
Y = np.full((len(SM_ALL), len(HEADS)), np.nan, np.float32)
for j, i in enumerate(ISO):
    Y[:n_ch, j] = meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float)
    Y[:n_ch, 4 + j] = AUX[:, j]
Y[n_ch:n_ch + n_pub, 8:13] = wide.to_numpy(float)

# Descriptors are computable for every row, so every row carries all ten.
d_block = np.full((len(SM_ALL), len(DNAMES)), np.nan, np.float32)
for r, s in enumerate(SM_ALL):
    m = mol(s)
    if m is None:
        continue
    for k, name in enumerate(DNAMES):
        try:
            d_block[r, k] = float(DESC[name](m))
        except Exception:
            pass
Y[:, len(ASSAY):] = d_block

# Normalisation from development rows only; using all rows would fit a statistic on
# held-out labels. Affine, so it cannot inflate a Spearman, but it should still be clean.
_norm = Y.copy()
_norm[np.flatnonzero(hold)] = np.nan
mu = np.nanmean(_norm, 0)
sd = np.nanstd(_norm, 0)
Yz_nan = ((Y - mu) / np.where(sd > 0, sd, 1.0)).astype(np.float32)
M = np.isfinite(Y)
print(f'table {Y.shape}: {n_ch} challenge + {n_pub} public + {n_nei} neighbours')
print('labels per head:', dict(zip(HEADS, M.sum(0).tolist())))


## The network

Identical to notebook 5 apart from the per-task weight vector in the loss.


In [ ]:
from chemprop import data, featurizers
from chemprop.models.model import MPNN

FZ = featurizers.SimpleMoleculeMolGraphFeaturizer()
HW = torch.tensor(HEAD_W, device=DEVICE)

# Targets live inside the datapoints, so chemprop's own loader batches them with the
# graphs and nothing has to be re-featurised per minibatch. NaN marks "not measured"
# and becomes the loss mask.
PTS_ALL = [data.MoleculeDatapoint.from_smi(s, Yz_nan[i])
           for i, s in enumerate(SM_ALL)]
PTS_TE = [data.MoleculeDatapoint.from_smi(s, np.zeros(len(HEADS), np.float32))
          for s in SM_TE]

def loader(idx=None, bs=128, shuffle=False, test=False):
    pts = PTS_TE if test else ([PTS_ALL[i] for i in idx] if idx is not None else PTS_ALL)
    return data.build_dataloader(data.MoleculeDataset(pts, FZ), num_workers=0,
                                 batch_size=bs, shuffle=shuffle, drop_last=False)

def _fp(m, bmg, V_d, X_d):
    # BatchMolGraph.to() moves in place and returns None, unlike a torch tensor, so it
    # must be called as a statement rather than used as an expression.
    if hasattr(bmg, 'to'):
        bmg.to(DEVICE)
    V_d = V_d.to(DEVICE) if torch.is_tensor(V_d) else V_d
    X_d = X_d.to(DEVICE) if torch.is_tensor(X_d) else X_d
    return m.fingerprint(bmg, V_d, X_d)

class Multi(nn.Module):
    def __init__(self, ckpt, n_out, warm=True):
        super().__init__()
        m = MPNN.load_from_file(ckpt)
        if not warm:
            for p in m.parameters():
                nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.zeros_(p)
        self.enc = m
        with torch.no_grad():                      # probe on CPU, before .to(DEVICE)
            bmg, V_d, X_d, *_ = next(iter(loader([0, 1, 2, 3], bs=4)))
            d = m.fingerprint(bmg, V_d, X_d).shape[1]
        self.head = nn.Sequential(nn.Linear(d, 512), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(512, n_out))

    def forward(self, bmg, V_d, X_d):
        return self.head(_fp(self.enc, bmg, V_d, X_d))

def train(tr_idx, epochs=25, warm=True, lr=3e-4, bs=128, seed=0):
    torch.manual_seed(seed)
    m = Multi(CK, len(HEADS), warm).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-5)
    ld = loader(tr_idx, bs=bs, shuffle=True)
    for ep in range(epochs):
        m.train()
        for batch in ld:
            bmg, V_d, X_d, Yb, *_ = batch
            t = Yb.to(DEVICE).float()
            msk = torch.isfinite(t)
            w = msk.float() * HW                   # per-task weight, zero where unmeasured
            den = w.sum()
            if den == 0:
                continue
            opt.zero_grad()
            p = m(bmg, V_d, X_d)
            loss = (((p - torch.nan_to_num(t)) ** 2) * w).sum() / den
            loss.backward()
            opt.step()
    return m

@torch.no_grad()
def predict(m, idx=None, bs=256, test=False):
    m.eval()
    out = [m(*batch[:3]).cpu().numpy() for batch in loader(idx, bs=bs, test=test)]
    return np.vstack(out) * sd + mu


## Smoke test

Two minutes. Confirms the warm start loads, the weighted loss runs, and the neighbour rows
batch without error, before committing to the full run.


In [ ]:
if (OUT / 'nb16_state.npz').exists():
    print('the full run has already started; skipping the smoke test')
else:
    y1 = meta['CYP3A4_pIC50_direct_inhibition'].to_numpy(float)
    obs = np.isfinite(y1)
    dev = np.flatnonzero(obs & ~hold)
    ho = np.flatnonzero(obs & hold)
    pub_idx = np.arange(n_ch, n_ch + n_pub)
    nei_idx = np.arange(n_ch + n_pub, n_ch + n_pub + n_nei)

    quick = np.concatenate([dev[:600], pub_idx[:2000], nei_idx[:1000]])
    t0 = time.time()
    m = train(quick, epochs=4, warm=True)
    p = predict(m, ho)[:, 3]
    print('CYP3A4 holdout rho %.4f after 4 epochs (%.0fs)'
          % (spearmanr(y1[ho], p).statistic, time.time() - t0))
    print('neighbour rows in the smoke batch:', len(nei_idx[:1000]))
    del m
    gc.collect()
    torch.cuda.empty_cache()
    print('\nsmoke test passed')


## Full run

Five grouped folds for out-of-fold predictions, one holdout model, one test model.
Neighbours and public qHTS rows are always in train, never in a validation fold.
Checkpointed after every fit, so re-running resumes.


In [ ]:
EPOCHS = 25
RESULTS = []
pub_idx = np.arange(n_ch, n_ch + n_pub)
nei_idx = np.arange(n_ch + n_pub, n_ch + n_pub + n_nei)
y_all = {i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
obs_any = np.isfinite(np.column_stack([y_all[i] for i in ISO])).any(1)
dev_rows = np.flatnonzero(obs_any & ~hold)
ho_rows = np.flatnonzero(obs_any & hold)
extra = np.concatenate([pub_idx, nei_idx]) if n_nei else pub_idx

CKPT = OUT / 'nb16_state.npz'
if CKPT.exists():
    st = np.load(CKPT, allow_pickle=True)
    oof_all = st['oof_all']
    done = set(st['done'].tolist())
    hp = st['hp'] if st['hp'].size else None
    tp = st['tp'] if st['tp'].size else None
    print('resuming; folds already done:', sorted(done),
          '| holdout', hp is not None, '| test', tp is not None)
else:
    oof_all = np.full((n_ch, 4), np.nan, np.float32)
    done, hp, tp = set(), None, None

def save():
    np.savez_compressed(CKPT, oof_all=oof_all, done=np.array(sorted(done)),
                        hp=hp if hp is not None else np.array([]),
                        tp=tp if tp is not None else np.array([]))

t0 = time.time()
for f, (tr, va) in enumerate(GroupKFold(5).split(dev_rows, dev_rows, G[dev_rows])):
    if f in done:
        continue
    m = train(np.concatenate([dev_rows[tr], extra]), epochs=EPOCHS)
    oof_all[dev_rows[va]] = predict(m, dev_rows[va])[:, :4]
    done.add(f)
    save()
    print(f'  fold {f} done, checkpointed ({time.time() - t0:.0f}s)', flush=True)
    del m
    gc.collect()
    torch.cuda.empty_cache()

if hp is None:
    m = train(np.concatenate([dev_rows, extra]), epochs=EPOCHS)
    hp = predict(m, ho_rows)[:, :4]
    save()
    print(f'  holdout model done ({time.time() - t0:.0f}s)', flush=True)
    del m
    gc.collect()
    torch.cuda.empty_cache()

if tp is None:
    mt = train(np.concatenate([dev_rows, ho_rows, extra]), epochs=EPOCHS)
    tp = predict(mt, test=True)[:, :4]
    save()
    print(f'  test model done ({time.time() - t0:.0f}s)', flush=True)
    del mt
    gc.collect()
    torch.cuda.empty_cache()

for j, i in enumerate(ISO):
    y = y_all[i]
    d = dev_rows[np.isfinite(y[dev_rows])]
    h = ho_rows[np.isfinite(y[ho_rows])]
    np.savez_compressed(OUT / f'nei_{i}.npz', oof=oof_all[d, j], dev=d,
                        hold=hp[np.isin(ho_rows, h), j], ho=h, test=tp[:, j])
    r = float(spearmanr(y[d], oof_all[d, j]).statistic)
    rh = float(spearmanr(y[h], hp[np.isin(ho_rows, h), j]).statistic)
    RESULTS.append(dict(iso=i, dev_spearman=r, holdout_spearman=rh, n_neighbours=int(n_nei)))
    json.dump(RESULTS, open(OUT / 'results_nb16.json', 'w'), indent=2)
    print(f'{i}: dev rho {r:.4f} | holdout rho {rh:.4f}', flush=True)

print('\nnotebook 5, same arms without neighbours: dev 0.5577 0.6935 0.4515 0.8022')
print('                                           macro dev 0.6262 | macro holdout 0.6652')
print('macro dev %.4f' % np.mean([x['dev_spearman'] for x in RESULTS]))
print('macro holdout %.4f' % np.mean([x['holdout_spearman'] for x in RESULTS]))

import shutil
shutil.make_archive('/content/nb16_results', 'zip', OUT)
print('results are on Drive at', OUT)
try:
    from google.colab import files
    files.download('/content/nb16_results.zip')
except Exception as e:
    print('browser download unavailable (%s); pull the zip off Drive instead'
          % type(e).__name__)
